# Day 1, afternoon: from words to digit strings

The same kind of **workbench** as this morning: you write no code here. Each section calls your package and
draws what comes out. The figures follow lecture L2.

| You implement | In | L2 slides | Sections here |
|---|---|---|---|
| `train_bigram` | `speech/gmm_hmm.py` | the bigram LM, with a silence word | 3, 5, 6 |
| `WordLoopDecoder._build` | `speech/gmm_hmm.py` | the word loop, the transition matrix | 3–6 |
| `WordLoopDecoder.decode_emissions` | `speech/gmm_hmm.py` | Viterbi, from the best path to words | 4–6 |

**Reference solution (since 17 November).** It is on branch `solutions` of the starter repository, not on
`main`, so pulling `main` never replaces your code. To run this notebook on it, or to compare with your own:
`git fetch course`, then `git checkout course/solutions -- <file>` for the file you want, or
`git diff HEAD course/solutions -- <file>` to compare. With your own code, every cell that reaches a function
you have not implemented says so and the notebook goes on.

As this morning, each function's body is one `raise NotImplementedError("TODO(d1): ...")`; its message, the
docstring and the L2 slide named in the table say what to write. (The "holes" in the plan below are the steps
of each function, in the order to write them.)

**Plan for the lab (2 h).**

| Time | What | Computer time |
|---|---|---|
| 0:00–0:10 | Sections 1–2 (provided code): run, look | a few s |
| 0:10–0:20 | `train_bigram` (2 holes) | |
| 0:20–0:40 | `_build` (3 holes); section 3 shows your matrix next to the grammar | |
| 0:40–1:05 | `decode_emissions` (2 holes); section 4 shows your trellis. Then start `--part connected` and `--part beam` in a terminal | about 2 min and 1 min |
| 1:05–1:35 | Sections 5–6: the $(\lambda,\beta)$ grid, matched vs mismatched text, beam width; their questions | about 1 min |
| 1:35–2:00 | Section 7; `pytest -m d1`; one `DECISIONS.md` entry; tag `d1` | |

Sections 1–2 use only provided code: run them first. A cell that reaches one of your unfinished functions
says which one; re-run it after editing the package (`autoreload` picks up your changes).

Data: `python scripts/fetch_data.py data` (or set `DSP_FSDD`). The word models are the ones of this morning
(5 states, 2 Gaussians, cached in `results/speech_classical/`).

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "speech").is_dir() else Path.cwd().parent  # the folder that contains speech/
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # results/ and data/ are relative to the bundle root

import jiwer
import librosa
from IPython.display import Audio, display

from asr_demo import digits  # provided: FSDD paths and cached word models (shared with the demo)
from speech import run_classical_baseline as R
from speech.gmm_hmm import WordLoopDecoder, add_noise, mfcc_features, train_bigram

SR = R.SR
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm, imported by a dependency


def not_yet(e):
    """What a cell prints when it reaches one of your stubs."""
    print(f"Not implemented yet: {e}\nImplement it in the package, then re-run this cell.")

**Knobs.** Change them and re-run the notebook. At 15 dB the acoustic evidence is weak, so the language model
and the search matter (L2, "What the bigram LM bought"); in clean speech almost everything works.

In [ ]:
SNR = 15        # signal-to-noise ratio of the test utterances in dB; None = clean
N_DEV = 30      # utterances to tune lambda and beta on
N_TEST = 60     # utterances to report on
SEED = 0

## 1. A connected utterance

The lab glues test recordings of single digits together, with silences of random length (50–250 ms) between
them, like `run_classical_baseline --part connected`. The digit strings come from a **grammar** in which each
digit has two likely successors (80 % of the probability). Here we also keep where each word is, for the
figures: the decoder will not know it.

In [ ]:
rng = np.random.default_rng(SEED)
data = R.load_fsdd(digits.FSDD)
pool = defaultdict(list)
for d in data:
    if d["idx"] < 5:  # test recordings only: the word models never saw them
        pool[d["digit"]].append(d["y"])
grammar = R.markov_chain(rng)  # row u: P(next digit | u); row 10: first digit


def utterance(seq, rng):
    """Like R.build_utterance, but also returns each word's span in frames (10 ms)."""
    def gap(s):
        return rng.normal(0, R.GAP_STD, int(SR * s))
    parts, spans, n = [gap(0.15)], [], int(SR * 0.15)
    for dgt in seq:
        w = pool[dgt][rng.integers(len(pool[dgt]))]
        g = gap(rng.uniform(0.05, 0.25))
        spans.append((R.WORDS[int(dgt)], n // 80, (n + len(w)) // 80))
        parts += [w, g]
        n += len(w) + len(g)
    return np.concatenate(parts), spans


def mark_words(ax, spans, top):
    """Dashed lines where the true words start and end, and the words above the plot."""
    for w, a, b in spans:
        for x in (a, b):
            ax.axvline(x - 0.5, color="k", ls="--", lw=0.7)
        ax.text((a + b) / 2, top, w, ha="center", va="bottom", fontsize=8, clip_on=False)


seq = R.sample_sequences(rng, grammar, 1, lmin=5, lmax=5)[0]
y, spans = utterance(seq, rng)
mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_fft=200, hop_length=80, n_mels=40))
fig, ax = plt.subplots(figsize=(13, 2.6))
ax.imshow(mel, origin="lower", aspect="auto", cmap="magma")
mark_words(ax, spans, 40)
ax.set(xlabel="frame $t$ (10 ms)", ylabel="mel band")
ax.set_title(f"reference: {R.to_words(seq)}", pad=14)
fig.tight_layout()
display(Audio(y, rate=SR))

## 2. What the word models see, frame by frame

The decoder works on the matrix of emission log-likelihoods $\log b_s(\mathbf{x}_t)$: one row per frame, one
column per state of every word model (10 digits with 5 states each, `sil` with 3: $|\mathcal{S}|=53$). Below, for each
word, the best of its states at each frame, relative to the best state overall (0 = the best word at that
frame). The red dots pick the best word **frame by frame**, with no model of how long words last.

In [ ]:
wm, norm = digits.word_models(5, 2)
VOCAB = R.DIGITS + ["sil"]
NAMES = R.WORDS + ["sil"]
sizes = np.array([wm.models[w].n_components for w in VOCAB])
offsets = np.concatenate([[0], np.cumsum(sizes)[:-1]])


def emissions(y, snr=None, rng=None):
    """(T, 53) log b_s(x_t): the same matrix as the provided WordLoopDecoder.emissions (usable before _build)."""
    x = norm(mfcc_features(add_noise(y, snr, rng) if snr is not None else y, SR))
    return np.hstack([wm.models[w]._compute_log_likelihood(x) for w in VOCAB])


B = emissions(y)
per_word = np.stack([B[:, o:o + n].max(1) for o, n in zip(offsets, sizes)], axis=1)  # (T, 11)
rel = per_word - per_word.max(1, keepdims=True)
fig, ax = plt.subplots(figsize=(13, 3))
ax.imshow(np.maximum(rel, -40).T, origin="lower", aspect="auto", cmap="viridis", interpolation="nearest")
ax.plot(per_word.argmax(1), "r.", ms=3)
mark_words(ax, spans, 10.6)
ax.set_yticks(range(11), NAMES)
ax.set_xlabel("frame $t$")
ax.set_title("best emission per word, minus the best overall (clipped at -40 nats)", pad=14)
fig.tight_layout()
frame_words = [NAMES[i] for i in per_word.argmax(1)]
changes = [w for i, w in enumerate(frame_words) if i == 0 or w != frame_words[i - 1]]
print(f"frame-by-frame best word changes {len(changes) - 1} times:", " ".join(changes[:40]), "...")

**What to see.** Inside a word the right model usually wins, but not at every frame, and the edges of words
are a mess. Picking the best word at every frame would output dozens of words. The word loop fixes this: a word
must go through its 5 states in order (it lasts), and leaving it costs $\lambda\log P(v\mid u)+\beta$.

## 3. Your decoding graph: `train_bigram` and `_build`

Your bigram LM is trained on 5000 digit strings drawn from the grammar (text only, no audio). Your `_build`
turns the 11 word HMMs and the LM into one $53\times53$ log transition matrix $\log A$ (L2, "The word-loop
transition matrix, block by block"). Left: the whole matrix (white = $-\infty$, impossible).
Middle: the word-entry block, $\lambda\log P(v\mid u)+\beta$ read from your matrix (here $\lambda=1$, $\beta=0$).
Right: the grammar that generated the text. If your LM learned it, the middle and right look alike.

In [ ]:
lm_text = R.sample_sequences(rng, grammar, 5000)
try:
    lm = train_bigram(lm_text, R.DIGITS)
except NotImplementedError as e:
    lm = None
    not_yet(e)

dec = None
if lm is not None:
    try:
        dec = WordLoopDecoder(wm, VOCAB, lm, lm_weight=1.0, word_penalty=0.0)
    except NotImplementedError as e:
        not_yet(e)

if dec is not None:
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("white")
    finals, firsts = offsets + sizes - 1, offsets
    exit_lp = np.log1p(-np.exp(dec.exit_logp))  # the part of an entry that is not the LM
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4), gridspec_kw={"width_ratios": [1.3, 1, 1]})
    im = axes[0].imshow(np.ma.masked_invalid(dec.logA), cmap=cmap, vmin=-8, vmax=0, interpolation="nearest")
    axes[0].set_xticks(offsets + sizes // 2, NAMES, rotation=90, fontsize=7)
    axes[0].set_yticks(offsets + sizes // 2, NAMES, fontsize=7)
    axes[0].set(title="your $\\log A$ (from state, to state)")
    fig.colorbar(im, ax=axes[0], shrink=0.8)
    entry = dec.logA[np.ix_(finals, firsts)] - exit_lp
    axes[1].imshow(entry[:10, :10], cmap="viridis", vmin=-6, vmax=0)
    axes[1].set(title="your entries $\\lambda\\log P(v\\mid u)+\\beta$", xlabel="next word $v$", ylabel="word $u$",
                xticks=range(10), yticks=range(10))
    axes[2].imshow(np.log(grammar[:10]), cmap="viridis", vmin=-6, vmax=0)
    axes[2].set(title="log of the true grammar", xlabel="next word $v$", ylabel="word $u$",
                xticks=range(10), yticks=range(10))
    fig.tight_layout()
    print("row u='sil' of your entries (a silence forgets the history):", np.round(entry[10], 1))

**Questions.** Why are the off-diagonal blocks empty except for one row each? Where does the silence word enter,
and why is $P(\text{sil}\mid u)$ the same for every $u$? What would a trigram change in this matrix
(L2, "The decoding graph grows with the $n$-gram order")?

## 4. Your Viterbi: `decode_emissions`

The provided `prune` sees $\delta_t$ at every frame (you pass it through `self.prune`). We listen to it to draw
the **trellis**: $\delta_t(s)$ for every state and frame, relative to the best state at that frame
(0 = best partial path, white = unreachable or pruned). Red dots: the best state (or word) at each frame. The
words are read from the *backtracked* path, which is not the same thing: the best state at frame $t$ may lose
later. Decoder: your bigram with $\lambda=16$, $\beta=-100$ (section 5 shows where these numbers come from).

Two views of the same trellis:
- **top, word level**, the whole utterance: for each word, its best state (11 rows);
- **bottom, state level**, a zoom on the frames between the white lines (`ZOOM`): the 5 states of every word.
  Inside the right word, the bright cells climb from state 0 to state 4 as the word unfolds.

In [ ]:
def decode_with_trellis(dec, B):
    """Run your decode_emissions and record delta at every frame through the provided prune."""
    frames, inner = [], dec.prune

    def recording(delta):
        out = inner(delta)
        frames.append(out.copy())
        return out
    dec.prune = recording  # an instance attribute hides the method, until `del`
    try:
        words = dec.decode_emissions(B)
    finally:
        del dec.prune
    if not frames:
        print("decode_emissions never called self.prune(delta): see its docstring (the beam needs it).")
    return words, np.array(frames)


def show_trellis(ax, D, spans, title, window=None, words=False):
    """D: (T, 53) recorded deltas. words=True: one row per word (its best state). window: (first, last) frame."""
    M = np.stack([D[:, o:o + n].max(1) for o, n in zip(offsets, sizes)], 1) if words else D
    a, b = window or (0, len(M))
    rel = M - M.max(1, keepdims=True)
    img = np.where(np.isfinite(rel), np.maximum(rel, -300), np.nan)[a:b]
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad("white")
    ax.imshow(np.ma.masked_invalid(img).T, origin="lower", aspect="auto", cmap=cmap, vmin=-300, vmax=0,
              interpolation="nearest", extent=(a - 0.5, b - 0.5, -0.5, M.shape[1] - 0.5))
    if words:
        ax.set_yticks(range(len(NAMES)), NAMES, fontsize=8)
    else:
        for o in offsets[1:]:
            ax.axhline(o - 0.5, color="0.6", lw=1.2)  # gray: visible on colours and on pruned (white) cells
        ax.set_yticks(offsets + sizes // 2, NAMES, fontsize=8)
        ax.set_yticks(np.arange(M.shape[1]), minor=True)
    ax.plot(np.arange(a, b), np.argmax(M, 1)[a:b], "r.", ms=3 if words else 4)
    mark_words(ax, [s for s in spans if a <= (s[1] + s[2]) / 2 < b], M.shape[1] - 0.5)
    ax.set_xlim(a - 0.5, b - 0.5)
    ax.set_xlabel("frame $t$")
    ax.set_title(title, pad=14, fontsize=10)


def first_words(spans, T, n=2):
    """A window over the first n words, with a margin."""
    return max(0, spans[0][1] - 8), min(T, spans[n - 1][2] + 8)


ZOOM = None  # (first frame, last frame) of the state-level view; None = the first two words


if dec is None:
    print("Skipped: this cell needs your train_bigram and WordLoopDecoder._build (section 3).")
if dec is not None:
    try:
        words, D = decode_with_trellis(WordLoopDecoder(wm, VOCAB, lm, lm_weight=16, word_penalty=-100.0), B)
        if len(D):
            zoom = ZOOM or first_words(spans, len(D))
            fig, axes = plt.subplots(2, 1, figsize=(13, 10), gridspec_kw={"height_ratios": [1, 2.4]})
            show_trellis(axes[0], D, spans, f"word level. Decoded: {R.to_words(words)}   "
                         f"(reference: {R.to_words(seq)})", words=True)
            for x in zoom:
                axes[0].axvline(x - 0.5, color="w", lw=2.5)
            show_trellis(axes[1], D, spans, f"state level, frames {zoom[0]}-{zoom[1]}", window=zoom)
            fig.tight_layout()
    except NotImplementedError as e:
        not_yet(e)

**Questions.** Is the output right? If not, which word competed with the right one, and where in the trellis?
Find a frame where the red dot is in another word than the one finally decoded: why is it not an error of
Viterbi (L2, "Viterbi is exact search; greedy search is not")? Change `SEED` for another utterance.

## 5. The LM weight $\lambda$ and the insertion term $\beta$

Now many utterances, at `SNR` dB. Each emission matrix is computed once; then every $(\lambda,\beta)$ is a new
decoding graph on the same matrices. The grid is tuned on the **dev** utterances only (L2, "Tune $\lambda$ and
$\beta$ on a development set"). $\lambda=0$ means no LM at all (only $\beta$ per word).

In [ ]:
def make_set(n, uniform=False):
    seqs = R.sample_sequences(rng, grammar, n, uniform=uniform)
    utts = [utterance(s, rng) for s in seqs]
    return [R.to_words(s) for s in seqs], [emissions(u, SNR, rng) for u, _ in utts], [sp for _, sp in utts]


t0 = time.perf_counter()
refs_dev, B_dev, _ = make_set(N_DEV)
refs_test, B_test, spans_test = make_set(N_TEST)
refs_mis, B_mis, _ = make_set(N_TEST, uniform=True)  # mismatched: digits drawn uniformly, the LM is wrong
print(f"{N_DEV + 2 * N_TEST} utterances at {SNR} dB in {time.perf_counter() - t0:.1f} s")


def evaluate(refs, Bs, lm_weight, penalty, **beam):
    """WER and error counts of your decoder, plus the search cost recorded by prune."""
    dec = WordLoopDecoder(wm, VOCAB, lm, lm_weight=lm_weight, word_penalty=penalty, **beam)
    hyps, scores = [], []
    for b in Bs:
        hyps.append(R.to_words(dec.decode_emissions(b)) or "<empty>")
        scores.append(dec.final_score)
    o = jiwer.process_words(refs, hyps)
    return dict(wer=o.wer, sub=o.substitutions, dele=o.deletions, ins=o.insertions, hyps=hyps,
                active=float(np.mean(dec.n_active)) if dec.n_active else np.nan, scores=np.array(scores))


LAMBDAS = (0, 1, 4, 16, 64)
if dec is None:
    print("Skipped: this cell needs your train_bigram and WordLoopDecoder._build (section 3).")
grid = None
if dec is not None:
    try:
        grid = {(lw, pen): evaluate(refs_dev, B_dev, lw, pen) for lw in LAMBDAS for pen in R.PENALTIES}
    except NotImplementedError as e:
        not_yet(e)

if grid is not None:
    W = np.array([[grid[lw, pen]["wer"] for pen in R.PENALTIES] for lw in LAMBDAS])
    best_lw, best_pen = min(grid, key=lambda k: grid[k]["wer"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw={"width_ratios": [1.4, 1]})
    axes[0].imshow(np.minimum(W, 1), cmap="Reds", vmin=0, vmax=1, aspect="auto")
    for i in range(len(LAMBDAS)):
        for j in range(len(R.PENALTIES)):
            axes[0].text(j, i, f"{100 * W[i, j]:.0f}", ha="center", va="center", fontsize=8,
                         weight="bold" if (LAMBDAS[i], R.PENALTIES[j]) == (best_lw, best_pen) else None)
    axes[0].set(title=f"dev WER (%) at {SNR} dB", xlabel="insertion term $\\beta$", ylabel="LM weight $\\lambda$",
                xticks=range(len(R.PENALTIES)), xticklabels=[f"{p:g}" for p in R.PENALTIES],
                yticks=range(len(LAMBDAS)), yticklabels=LAMBDAS)
    for key, lab in (("sub", "substitutions"), ("dele", "deletions"), ("ins", "insertions")):
        axes[1].plot(range(len(R.PENALTIES)), [grid[best_lw, p][key] for p in R.PENALTIES], "o-", label=lab)
    axes[1].set(title=f"errors on dev at $\\lambda$={best_lw}", xlabel="insertion term $\\beta$",
                xticks=range(len(R.PENALTIES)), xticklabels=[f"{p:g}" for p in R.PENALTIES], ylabel="count")
    axes[1].legend()
    fig.tight_layout()
    print(f"best on dev: lambda={best_lw}, beta={best_pen:g}, dev WER {100 * grid[best_lw, best_pen]['wer']:.1f} %")

Now the **test** utterances, once, with the dev choices: the best $\beta$ without LM against the best
$(\lambda,\beta)$ with your bigram, on text from the grammar (matched) and on uniformly random digits
(mismatched: the LM's prior is wrong).

In [ ]:
if grid is None:
    print("Skipped: this cell needs the (lambda, beta) grid above, i.e. your decode_emissions.")
if grid is not None:
    pen0 = min(R.PENALTIES, key=lambda p: grid[0, p]["wer"])
    table = {}
    for dom, refs, Bs in (("matched", refs_test, B_test), ("mismatched", refs_mis, B_mis)):
        table[dom, "no LM"] = evaluate(refs, Bs, 0, pen0)["wer"]
        table[dom, "bigram"] = evaluate(refs, Bs, best_lw, best_pen)["wer"]
    fig, ax = plt.subplots(figsize=(6, 3))
    for k, sys_ in enumerate(("no LM", "bigram")):
        vals = [100 * table[dom, sys_] for dom in ("matched", "mismatched")]
        bars = ax.bar(np.arange(2) + 0.38 * k, vals, 0.38, label=sys_ if sys_ == "no LM" else
                      f"bigram ($\\lambda$={best_lw}, $\\beta$={best_pen:g})")
        ax.bar_label(bars, fmt="%.1f", fontsize=8)
    ax.set(xticks=np.arange(2) + 0.19, xticklabels=["matched text", "mismatched text"], ylabel="test WER (%)",
           title=f"{N_TEST} test utterances at {SNR} dB")
    ax.legend(fontsize=8)
    fig.tight_layout()

**Questions.**
- Along a row of the heatmap, which error type does $\beta$ trade against which? Why is its scale hundreds of
  nats and not 1 (L2, "The insertion term $\beta$, and its scale")?
- Why does a large $\lambda$ also need a smaller penalty (look at the diagonal of good cells)?
- On mismatched text, is the bigram still worth it? What does that say about deploying an LM trained on the
  wrong domain?

## 6. Beam search: accuracy against search cost

Exact Viterbi keeps all 53 states alive at every frame. The provided `prune` keeps fewer (L2, "Beam search: two
rules"): a **score beam** $\Omega$ (keep states within $\Omega$ nats of the best) or a **top-N beam** $N_{beam}$
(keep the $N_{beam}$ best; $N_{beam}=1$ is greedy search). Cost = average number of active states per frame.
A **search error**: the pruned decoder ends with a lower score than exact search, under the same model.

In [ ]:
OMEGAS = (np.inf, 600, 400, 300, 250, 200, 150, 100, 50)
N_BEAMS = (32, 16, 8, 4, 2, 1)
if grid is None:
    print("Skipped: this cell needs the (lambda, beta) grid above, i.e. your decode_emissions.")
beam = None
if grid is not None:
    exact = evaluate(refs_test, B_test, best_lw, best_pen)
    beam = []
    for rule, kw in [("score", dict(beam=o)) for o in OMEGAS] + [("top-N", dict(n_beam=n)) for n in N_BEAMS]:
        r = evaluate(refs_test, B_test, best_lw, best_pen, **kw)
        r.update(rule=rule, label="∞" if kw.get("beam") == np.inf else f"{kw.get('beam', kw.get('n_beam')):g}",
                 search_err=float(np.mean(r["scores"] < exact["scores"] - 1e-6)))
        beam.append(r)
    fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
    for rule, mk in (("score", "o-"), ("top-N", "s-")):
        rs = [r for r in beam if r["rule"] == rule]
        for ax, key, lab in ((axes[0], "wer", "test WER (%)"), (axes[1], "search_err", "search errors (% utterances)")):
            ax.plot([r["active"] for r in rs], [100 * r[key] for r in rs], mk,
                    label="score beam $\\Omega$" if rule == "score" else "top-N beam $N_{beam}$")
            last = np.inf
            for r in sorted(rs, key=lambda r: -r["active"]):  # label a point only if not crowded
                if r["active"] < 0.85 * last:
                    ax.annotate(r["label"], (r["active"], 100 * r[key]), fontsize=7, xytext=(3, 3),
                                textcoords="offset points")
                    last = r["active"]
            ax.set(xscale="log", xlabel="active states per frame (log scale)", ylabel=lab)
    axes[0].set_title(f"$\\lambda$={best_lw}, $\\beta$={best_pen:g}, {SNR} dB")
    axes[0].legend()
    fig.tight_layout()
    cost = [best_lw * np.log(lm[u][v]) + best_pen for u in R.DIGITS for v in R.DIGITS]
    print(f"entering a digit costs lambda*log P(v|u) + beta = {max(cost):.0f} to {min(cost):.0f} nats")

**What to see.** Both rules keep the exact WER with far fewer than 53 states, then the WER climbs fast. For the
score beam, compare $\Omega$ with the cost of entering a word printed above: when $\Omega$ is smaller, a path
that has just entered a new word falls out of the beam, however good the word is (L2, "Measured: the beam width
is an accuracy–cost knob"). Below: a test utterance that exact search gets right and greedy search gets wrong,
at state level on its first two words, with the pruned states in white (titles: the whole decoded output).

In [ ]:
if beam is None:
    print("Skipped: this cell needs the (lambda, beta) grid above, i.e. your decode_emissions.")
if beam is not None:
    greedy = next(r for r in beam if r["rule"] == "top-N" and r["label"] == "1")
    diff = [i for i, (a, b) in enumerate(zip(exact["hyps"], greedy["hyps"])) if a != b]
    pure = [i for i in diff if exact["hyps"][i] == refs_test[i]]  # exact search right, greedy wrong
    if not diff:
        print("Greedy search found the exact output on every test utterance here: try SNR = 10.")
    else:
        i = (pure or diff)[0]
        zoom = first_words(spans_test[i], len(B_test[i]))
        fig, axes = plt.subplots(1, 3, figsize=(15, 7), sharey=True)
        for ax, (title, kw) in zip(axes, (("exact", {}), ("score beam $\\Omega$=250", dict(beam=250)),
                                           ("greedy, $N_{beam}$=1", dict(n_beam=1)))):
            d = WordLoopDecoder(wm, VOCAB, lm, lm_weight=best_lw, word_penalty=best_pen, **kw)
            words, D = decode_with_trellis(d, B_test[i])
            show_trellis(ax, D, spans_test[i], f"{title}\n{R.to_words(words) or '<empty>'}", window=zoom)
        fig.tight_layout()
        print(f"test utterance {i}, reference: {refs_test[i]}.",
              "Exact search is right: greedy makes a search error." if pure else
              "Exact search is wrong too here (a model error), and greedy adds search errors.")

**Questions.**
- Which beam would you deploy, and how much does it save against exact search? Is the saving worth it for
  53 states? Would it be for a graph of 50 000 states (a large-vocabulary decoder)?
- When the beam WER is bad, are the errors search errors or model errors? Which one does a wider beam fix
  (L2, "Search error or model error?")?

## 7. Your benchmark runs, as figures

The graded numbers come from the scripts, on more utterances and every SNR:
```
export DSP_MILESTONE=d1
python -m speech.run_classical_baseline --fsdd data/fsdd/recordings --part connected
python -m speech.run_classical_baseline --fsdd data/fsdd/recordings --part beam
```
This cell draws their CSVs (if they exist): test WER against $\lambda$ for every condition, and the beam
results as an accuracy–cost Pareto front, with your `pareto_mask_array` from this morning.

In [ ]:
import pandas as pd

from common.pareto import pareto_mask_array

con, bm = ROOT / "results/speech_classical/connected_digits_lm.csv", ROOT / "results/speech_classical/beam_search.csv"
if not con.exists() or not bm.exists():
    missing = " and ".join(f.name for f in (con, bm) if not f.exists())
    print(f"No {missing} yet: run the commands above, then re-run this cell.")
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8)) if con.exists() or bm.exists() else (None, [None, None])
if con.exists():
    df = pd.read_csv(con)
    for (dom, snr), g in df[df.lm_weight > 0].groupby(["domain", "snr"], sort=False):
        line, = axes[0].plot(g.lm_weight, 100 * g.wer, "o-" if dom == "matched" else "x--", ms=4, label=f"{dom}, {snr}")
        a = df[(df.domain == dom) & (df.snr == snr) & (df.lm_weight == 0)]
        if len(a):
            axes[0].axhline(100 * a.wer.iloc[0], color=line.get_color(), lw=0.6, ls=":")
    axes[0].set(xscale="log", xlabel="LM weight $\\lambda$", ylabel="test WER (%)",
                title="--part connected (dotted: acoustic only)")
    axes[0].legend(fontsize=7, ncol=2)
if bm.exists():
    df = pd.read_csv(bm)
    for snr, g in df.groupby("snr", sort=False):
        pts = axes[1].plot(g.active_states, 100 * g.wer, "o", ms=4, alpha=0.5, label=f"{snr}")[0]
        try:
            front = pareto_mask_array(g[["active_states", "wer"]].to_numpy(), ["min", "min"])
            f = g[front].sort_values("active_states")
            axes[1].step(f.active_states, 100 * f.wer, where="post", color=pts.get_color())
        except NotImplementedError as e:
            not_yet(e)
    axes[1].set(xscale="log", yscale="log", xlabel="active states per frame", ylabel="test WER (%)",
                title="--part beam: points and Pareto front")
    axes[1].legend(fontsize=7, title="SNR")
if fig is not None:
    fig.tight_layout()

## Before you leave

Write at least one `DECISIONS.md` entry for the afternoon (format in `SUBMISSION_SPEC.md`), for example the
$(\lambda,\beta)$ you would deploy at 15 dB, or the beam width you choose, with the rows or the figure that
justify it. Then try your decoder live: `python -m asr_demo.server`, entries `digits/3-word-loop` and
`digits/4-word-loop+bigram`.